In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [2]:
df = pd.read_csv("OnlineRetail.csv", encoding="latin1")

print(df.head())

print(df.info())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
4  12/1/2010 8:26       3.39     17850.0  United Kingdom  
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non

In [3]:
df = df.dropna(subset=["CustomerID"]).copy()

df = df[~df["InvoiceNo"].astype(str).str.startswith("C")].copy()

df["TotalSales"] = df["Quantity"] * df["UnitPrice"]

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

df = df.dropna(subset=["InvoiceDate"])

print(df.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  TotalSales  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom       15.30  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom       22.00  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34  


Task 1: Find unique UK customers

In [4]:
uk_data = df[df["Country"] == "United Kingdom"]

unique_uk_customers = uk_data["CustomerID"].nunique()

print("Total UK Transactions:", len(uk_data))
print("Unique UK Customers:", unique_uk_customers)

Total UK Transactions: 354345
Unique UK Customers: 3921


Task 2: Find the top 3 countries by total sales

In [5]:
country_sales = df.groupby("Country")["TotalSales"].sum()

country_sales = country_sales.sort_values(
    ascending=False
)

print("Total Sales by Country:")
print(country_sales)


top_3_countries = country_sales.head(3)

print("\nTop 3 Countries by Total Sales:")
print(top_3_countries)

Total Sales by Country:
Country
United Kingdom          7308391.554
Netherlands              285446.340
EIRE                     265545.900
Germany                  228867.140
France                   209024.050
Australia                138521.310
Spain                     61577.110
Switzerland               56443.950
Belgium                   41196.340
Sweden                    38378.330
Japan                     37416.370
Norway                    36165.440
Portugal                  33439.890
Finland                   22546.080
Singapore                 21279.290
Channel Islands           20450.440
Denmark                   18955.340
Italy                     17483.240
Cyprus                    13590.380
Austria                   10198.680
Poland                     7334.650
Israel                     7221.690
Greece                     4760.520
Iceland                    4310.000
Canada                     3666.380
USA                        3580.390
Malta                      2725.

Task 3: Calculate average order value and highlight orders above 10,000

In [6]:
order_values = df.groupby("InvoiceNo").agg(
    CustomerID=("CustomerID", "first"),
    Country=("Country", "first"),
    OrderValue=("TotalSales", "sum")
).reset_index()

average_order_value = order_values["OrderValue"].mean()

print("Average Order Value:", average_order_value)

high_value_orders = order_values[
    order_values["OrderValue"] > 10000
]

print("\nOrders Above 10,000:")
print(high_value_orders)

Average Order Value: 480.7621873111782

Orders Above 10,000:
      InvoiceNo  CustomerID         Country  OrderValue
564      537659     18102.0  United Kingdom    15885.49
1659     540689     17450.0  United Kingdom    12797.52
1692     540815     15749.0  United Kingdom    15160.90
1815     541206     14646.0     Netherlands    10389.06
1824     541220     14156.0            EIRE    16774.72
1909     541431     12346.0  United Kingdom    77183.60
2843     543989     12415.0       Australia    14022.92
3076     544479     14646.0     Netherlands    11956.10
3494     545475     12415.0       Australia    16726.84
4017     546789     15769.0  United Kingdom    10065.00
4387     547708     17450.0  United Kingdom    11343.20
4541     548011     14646.0     Netherlands    15719.56
5463     550461     15749.0  United Kingdom    21535.90
6437     552883     14646.0     Netherlands    14415.74
6731     553546     12415.0       Australia    11924.80
7722     555920     18102.0  United Kingdom

Task 4: Perform a t-test between France and Germany

In [7]:
france_orders = order_values[
    order_values["Country"] == "France"
]["OrderValue"]

germany_orders = order_values[
    order_values["Country"] == "Germany"
]["OrderValue"]

print("France Average Order Value:", france_orders.mean())
print("Germany Average Order Value:", germany_orders.mean())

t_statistic, p_value = stats.ttest_ind(
    france_orders,
    germany_orders,
    equal_var=False
)

# Display results
print("\nT-test Results")
print("T-statistic:", t_statistic)
print("P-value:", p_value)

if p_value < 0.05:
    print("The difference is statistically significant.")
else:
    print("The difference is not statistically significant.")

France Average Order Value: 537.3368894601543
Germany Average Order Value: 500.8033698030635

T-test Results
T-statistic: 0.8117525375831979
P-value: 0.41718050073965024
The difference is not statistically significant.


Task 5: Generate a summary of customer purchasing patterns

In [8]:
top_countries = country_sales.head(3)

print("Top 3 Countries:")
print(top_countries)

print("\nAverage Order Value:")
print(order_values["OrderValue"].mean())

invoice_quantity = df.groupby("InvoiceNo")["Quantity"].sum()

print("\nAverage Items Per Order:")
print(invoice_quantity.mean())

df["Month"] = df["InvoiceDate"].dt.to_period("M")

monthly_sales = df.groupby("Month")["TotalSales"].sum()

print("\nMonthly Sales:")
print(monthly_sales)

best_month = monthly_sales.idxmax()

print("\nHighest Sales Month:", best_month)

Top 3 Countries:
Country
United Kingdom    7308391.554
Netherlands        285446.340
EIRE               265545.900
Name: TotalSales, dtype: float64

Average Order Value:
480.7621873111782

Average Items Per Order:
279.5476909797151

Monthly Sales:
Month
2010-12     572713.890
2011-01     569445.040
2011-02     447137.350
2011-03     595500.760
2011-04     469200.361
2011-05     678594.560
2011-06     661213.690
2011-07     600091.011
2011-08     645343.900
2011-09     952838.382
2011-10    1039318.790
2011-11    1161817.380
2011-12     518192.790
Freq: M, Name: TotalSales, dtype: float64

Highest Sales Month: 2011-11


Customer Purchasing Insights
1. High-value countries: The top three countries by total sales are [Country 1], [Country 2], and [Country 3]. These countries contribute the highest revenue in the analyzed dataset.
2. Customer spending: The average order value is [Amount]. This helps understand how much customers typically spend per invoice.
3. Order quantity: Customers purchase an average of [Amount] items per invoice, which gives an overview of purchasing behavior.
4. Seasonal trends: The highest sales were recorded in [Month]. Monthly sales analysis can help identify busy shopping periods.
5. Business recommendations: The company can focus on high-revenue countries, encourage repeat purchases, and plan promotional campaigns during periods with lower sales.